# Universe Checklist Workflow

This notebook shows how to apply universe/checklist masks consistently in factor analysis and backtesting.

## Build Universe and Checklist

We first create a small synthetic multi-symbol dataset, then define metadata/tags and mask rules.

In [ ]:
import polars as pl

from factorium import AggBar, Checklist, Universe
from factorium.backtest import Backtester
from factorium.universe import ExcludeStablecoins, MinVolume, TagFilter


In [ ]:
DAY_MS = 86_400_000
BASE_TS = 1_700_000_000_000

rows = []
for i in range(30):
    ts = BASE_TS + i * DAY_MS
    rows.extend(
        [
            {"start_time": ts, "end_time": ts + 3_600_000, "symbol": "BTCUSDT", "open": 100 + i, "high": 101 + i, "low": 99 + i, "close": 100 + i, "volume": 20_000 + i, "alpha": float(i + 1)},
            {"start_time": ts, "end_time": ts + 3_600_000, "symbol": "USDCUSDT", "open": 1.0, "high": 1.0, "low": 1.0, "close": 1.0, "volume": 50_000, "alpha": float(50 - i)},
            {"start_time": ts, "end_time": ts + 3_600_000, "symbol": "NEWUSDT", "open": 10 + i, "high": 11 + i, "low": 9 + i, "close": 10 + i, "volume": 500 + i, "alpha": float(100 + i)},
        ]
    )

agg = AggBar(pl.DataFrame(rows))

metadata = {
    "BTCUSDT": {"symbol": "BTCUSDT", "base_asset": "BTC", "quote_asset": "USDT", "status": "TRADING", "listing_date": BASE_TS - 365 * DAY_MS},
    "USDCUSDT": {"symbol": "USDCUSDT", "base_asset": "USDC", "quote_asset": "USDT", "status": "TRADING", "listing_date": BASE_TS - 365 * DAY_MS, "is_stablecoin_pair": True},
    "NEWUSDT": {"symbol": "NEWUSDT", "base_asset": "NEW", "quote_asset": "USDT", "status": "TRADING", "listing_date": BASE_TS - 30 * DAY_MS},
}
tags = {"BTC": ["layer1"], "USDC": ["stablecoin"], "NEW": ["meme"]}

universe = Universe([ExcludeStablecoins()])
checklist = Checklist([TagFilter(include=["layer1", "meme"]), MinVolume(window=5, threshold=1_000)])


## Apply Mask to AggBar

Use `AggBar.with_mask(...)` to add boolean columns that can be reused downstream.

In [ ]:
agg = agg.with_mask(name="in_universe", mask_source=universe, metadata=metadata, tags=tags)
agg = agg.with_mask(name="in_checklist", mask_source=checklist, metadata=metadata, tags=tags)

agg.data[["symbol", "in_universe", "in_checklist"]].head(9)


## Evaluate Factor with Mask

Use the same mask in `Factor.eval(..., mask=...)` so ranking/evaluation happens only inside your tradable universe.

In [ ]:
factor = agg["alpha"]
masked_result = factor.eval(prices=agg, periods=1, quantiles=2, mask="in_checklist")
unmasked_result = factor.eval(prices=agg, periods=1, quantiles=2)

print("factor:", masked_result.factor_name)
masked_rows = sum(len(df) for df in masked_result.quantile_returns.values())
unmasked_rows = sum(len(df) for df in unmasked_result.quantile_returns.values())
print("masked quantile rows:", masked_rows)
print("unmasked quantile rows:", unmasked_rows)


## Run Backtest with Mask

Finally, pass `mask=` into `Backtester(...)` to keep positions inside the same universe constraints.

In [ ]:
try:
    bt = Backtester(
        prices=agg,
        signal=factor,
        holding_period=3,
        neutralization="market",
        mask="in_universe",
    )
    bt_result = bt.run()
    bt_result.metrics
except Exception as exc:
    print("Backtest run needs enough cross-sectional signals in each bar:", exc)
